# Lightweight GIF + Short Video Creator (Channel Upload Edition)

This notebook is a **simple, upload-focused workflow** for turning either:

- a **folder of images** into a small GIF and/or MP4, or
- an **existing video** into a shorter, lighter GIF and/or MP4.

## What this notebook does
1. Loads images or a source video
2. Trims the clip to a short duration
3. Resizes for lighter uploads
4. Supports **landscape**, **square**, **portrait**, or **original** aspect presets
5. Optionally center-crops or letterboxes
6. Optionally adds top/bottom text
7. Exports a **small GIF**
8. Exports a **small MP4**
9. Previews the outputs inside the notebook

---

### Good starter presets
- **Square promo / channel post:** `ASPECT_PRESET = "square"`
- **Regular video:** `ASPECT_PRESET = "landscape"`
- **Shorts / reels style:** `ASPECT_PRESET = "portrait"`

> The notebook is structured like your original one: intro, configuration, dependency checks, helper functions, preview, then export.

## Install once if needed

Run this only if the imports cell says something is missing:

```python
%pip install pillow imageio imageio-ffmpeg matplotlib numpy
```

`imageio-ffmpeg` is the easiest way to enable MP4 writing from inside Jupyter.

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
from pathlib import Path

# Choose one mode:
#   - "images_to_gif"
#   - "images_to_mp4"
#   - "images_to_both"
#   - "video_to_gif"
#   - "video_to_mp4"
#   - "video_to_both"
MODE = "images_to_both"

# Inputs
INPUT_DIR = Path("input_frames")     # folder of images for image-based modes
FILE_GLOB = "*.png"                  # change to *.jpg, *.jpeg, *.webp, etc.
INPUT_VIDEO = Path("input.mp4")      # source video for video-based modes

# Output
OUT_DIR = Path("outputs_media")
OUT_DIR.mkdir(parents=True, exist_ok=True)
BASENAME = "channel_clip"

# Clip length / frame limits
MAX_SECONDS = 8                       # keep clips short and upload-friendly
MAX_FRAMES = 120                      # safety cap

# Export speeds
GIF_FPS = 10
VIDEO_FPS = 24

# Size / format presets
#   - "original"   -> keep source aspect ratio
#   - "landscape"  -> 16:9
#   - "square"     -> 1:1
#   - "portrait"   -> 9:16
ASPECT_PRESET = "square"

# Main export widths (heights are derived from aspect preset)
GIF_WIDTH = 480
VIDEO_WIDTH = 720

# Resize behavior
CENTER_CROP = True                    # True = fill frame by cropping, False = pad with borders
PAD_COLOR = (0, 0, 0)

# Optional text overlays
TOP_TEXT = None                       # example: "New upload"
BOTTOM_TEXT = None                    # example: "Subscribe"
TEXT_MARGIN = 18

# Optional motion style
INCLUDE_REVERSE_BOUNCE = False        # creates forward + reverse loop from same frames

# GIF behavior
GIF_LOOP = 0                          # 0 = loop forever
GIF_OPTIMIZE = True

# MP4 behavior
MP4_CODEC = "libx264"
MP4_QUALITY = 7                       # imageio quality scale is usually 1..10

print("MODE       :", MODE)
print("INPUT_DIR  :", INPUT_DIR.resolve())
print("INPUT_VIDEO:", INPUT_VIDEO.resolve())
print("OUT_DIR    :", OUT_DIR.resolve())

In [ ]:
# -----------------------------
# Imports + optional dependencies
# -----------------------------
import math
import os
import re
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import imageio.v2 as imageio
    HAS_IMAGEIO = True
except Exception:
    HAS_IMAGEIO = False

try:
    import imageio_ffmpeg  # noqa: F401
    HAS_IMAGEIO_FFMPEG = True
except Exception:
    HAS_IMAGEIO_FFMPEG = False

try:
    from PIL import Image, ImageOps, ImageDraw, ImageFont
    HAS_PIL = True
except Exception:
    HAS_PIL = False

try:
    from IPython.display import Image as IPyImage, Video as IPyVideo, display
    HAS_IPYTHON_DISPLAY = True
except Exception:
    HAS_IPYTHON_DISPLAY = False

print("HAS_IMAGEIO       :", HAS_IMAGEIO)
print("HAS_IMAGEIO_FFMPEG:", HAS_IMAGEIO_FFMPEG)
print("HAS_PIL           :", HAS_PIL)
print("HAS_FFMPEG_BIN    :", shutil.which("ffmpeg") is not None)
print("HAS_DISPLAY       :", HAS_IPYTHON_DISPLAY)

if not HAS_PIL:
    raise RuntimeError("Pillow is required for this notebook. Install with: %pip install pillow")

if not HAS_IMAGEIO:
    print("⚠️ imageio is missing. GIF/MP4 export will not work until installed.")

try:
    RESAMPLE_LANCZOS = Image.Resampling.LANCZOS
except AttributeError:
    RESAMPLE_LANCZOS = Image.LANCZOS

In [ ]:
# -----------------------------
# Helpers
# -----------------------------
def natural_key(text):
    text = str(text)
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', text)]


def is_video_mode(mode: str) -> bool:
    return mode.startswith("video_")


def wants_gif(mode: str) -> bool:
    return mode.endswith("_gif") or mode.endswith("_both")


def wants_mp4(mode: str) -> bool:
    return mode.endswith("_mp4") or mode.endswith("_both")


def list_input_images(input_dir: Path, pattern: str):
    files = list(input_dir.glob(pattern))
    files = [p for p in files if p.is_file()]
    files.sort(key=natural_key)
    return files


def pil_from_array(arr: np.ndarray) -> Image.Image:
    arr = np.asarray(arr)
    if arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    if arr.ndim == 2:
        return Image.fromarray(arr, mode="L").convert("RGB")
    if arr.ndim == 3 and arr.shape[2] == 4:
        return Image.fromarray(arr, mode="RGBA").convert("RGB")
    return Image.fromarray(arr).convert("RGB")


def load_image_frames(input_dir: Path, pattern: str):
    files = list_input_images(input_dir, pattern)
    if not files:
        raise FileNotFoundError(f"No files matched {pattern!r} inside {input_dir}")
    frames = [Image.open(p).convert("RGB") for p in files]
    print(f"Loaded {len(frames)} image frame(s)")
    return frames, files


def read_video_frames(video_path: Path, target_sample_fps: int, max_seconds: float, max_frames: int):
    if not HAS_IMAGEIO:
        raise RuntimeError("imageio is required to read video files")
    if not video_path.exists():
        raise FileNotFoundError(video_path)

    reader = imageio.get_reader(str(video_path))
    meta = {}
    try:
        meta = reader.get_meta_data()
    except Exception:
        meta = {}

    src_fps = float(meta.get("fps", target_sample_fps or 12))
    stride = max(1, int(round(src_fps / max(1, target_sample_fps))))
    max_src_frames = int(max_seconds * src_fps) if max_seconds else None

    frames = []
    used_indices = []
    for idx, frame in enumerate(reader):
        if max_src_frames is not None and idx >= max_src_frames:
            break
        if idx % stride != 0:
            continue
        frames.append(pil_from_array(frame))
        used_indices.append(idx)
        if len(frames) >= max_frames:
            break

    try:
        reader.close()
    except Exception:
        pass

    if not frames:
        raise RuntimeError("No frames were extracted from the video")

    print(f"Video source fps: {src_fps:.2f}")
    print(f"Stride used      : every {stride} source frame(s)")
    print(f"Extracted frames : {len(frames)}")
    return frames, {"src_fps": src_fps, "used_indices": used_indices, "meta": meta}


def aspect_size_from_width(width: int, aspect_preset: str, original_size=None):
    if aspect_preset == "landscape":
        return (int(width), int(round(width * 9 / 16)))
    if aspect_preset == "square":
        return (int(width), int(width))
    if aspect_preset == "portrait":
        return (int(width), int(round(width * 16 / 9)))
    if aspect_preset == "original":
        if original_size is None:
            raise ValueError("original_size is required when ASPECT_PRESET='original'")
        ow, oh = original_size
        h = int(round(width * (oh / ow)))
        return (int(width), max(1, h))
    raise ValueError(f"Unknown aspect preset: {aspect_preset}")


def fit_or_pad(img: Image.Image, size, center_crop=True, pad_color=(0, 0, 0)):
    if center_crop:
        return ImageOps.fit(img, size, method=RESAMPLE_LANCZOS, centering=(0.5, 0.5))
    contained = ImageOps.contain(img, size, method=RESAMPLE_LANCZOS)
    canvas = Image.new("RGB", size, color=pad_color)
    x = (size[0] - contained.size[0]) // 2
    y = (size[1] - contained.size[1]) // 2
    canvas.paste(contained, (x, y))
    return canvas


def add_text_overlay(img: Image.Image, top_text=None, bottom_text=None, margin=18):
    if not top_text and not bottom_text:
        return img

    out = img.copy()
    draw = ImageDraw.Draw(out)
    font = ImageFont.load_default()
    w, h = out.size

    def draw_outlined_text(x, y, text, anchor):
        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(1,1),(-1,1),(1,-1)]:
            draw.text((x+dx, y+dy), text, font=font, fill=(0,0,0), anchor=anchor)
        draw.text((x, y), text, font=font, fill=(255,255,255), anchor=anchor)

    if top_text:
        draw_outlined_text(w // 2, margin, top_text, anchor="ma")
    if bottom_text:
        draw_outlined_text(w // 2, h - margin, bottom_text, anchor="md")

    return out


def ensure_even_size(img: Image.Image):
    w, h = img.size
    w2 = w - (w % 2)
    h2 = h - (h % 2)
    if (w2, h2) == (w, h):
        return img
    return img.crop((0, 0, w2, h2))


def to_numpy_rgb(frames_pil):
    return [np.asarray(f.convert("RGB"), dtype=np.uint8) for f in frames_pil]


def maybe_bounce(frames):
    if not INCLUDE_REVERSE_BOUNCE or len(frames) < 3:
        return frames
    return frames + frames[-2:0:-1]


def preprocess_frames(frames_pil, target_width: int, aspect_preset: str, center_crop=True, pad_color=(0,0,0), top_text=None, bottom_text=None, ensure_even=False):
    if not frames_pil:
        raise ValueError("No frames provided")
    size = aspect_size_from_width(target_width, aspect_preset, original_size=frames_pil[0].size)
    processed = []
    for f in frames_pil:
        img = fit_or_pad(f.convert("RGB"), size, center_crop=center_crop, pad_color=pad_color)
        img = add_text_overlay(img, top_text=top_text, bottom_text=bottom_text, margin=TEXT_MARGIN)
        if ensure_even:
            img = ensure_even_size(img)
        processed.append(img)
    return processed


def write_gif(frames_pil, output_path: Path, fps=10, loop=0, optimize=True):
    if not frames_pil:
        raise ValueError("No GIF frames to write")
    duration_ms = int(round(1000 / max(1, fps)))
    frames_pil[0].save(
        output_path,
        save_all=True,
        append_images=frames_pil[1:],
        duration=duration_ms,
        loop=loop,
        optimize=optimize,
        disposal=2,
    )
    return output_path


def write_mp4(frames_pil, output_path: Path, fps=24, codec="libx264", quality=7):
    if not HAS_IMAGEIO:
        raise RuntimeError("imageio is required for MP4 writing")
    frames_pil = [ensure_even_size(f.convert("RGB")) for f in frames_pil]
    frames_np = to_numpy_rgb(frames_pil)

    writer = None
    last_error = None
    for kwargs in [
        dict(fps=fps, codec=codec, quality=quality, pixelformat="yuv420p", macro_block_size=None),
        dict(fps=fps, codec=codec, pixelformat="yuv420p", macro_block_size=None),
        dict(fps=fps),
    ]:
        try:
            writer = imageio.get_writer(str(output_path), format="FFMPEG", **kwargs)
            for frame in frames_np:
                writer.append_data(frame)
            writer.close()
            return output_path
        except Exception as e:
            last_error = e
            try:
                if writer is not None:
                    writer.close()
            except Exception:
                pass
            writer = None
    raise RuntimeError(f"MP4 export failed. Install imageio-ffmpeg or ffmpeg. Last error: {last_error}")


print("Helpers ready.")

In [ ]:
# -----------------------------
# Load source frames
# -----------------------------
if is_video_mode(MODE):
    raw_frames, source_info = read_video_frames(
        INPUT_VIDEO,
        target_sample_fps=max(GIF_FPS, VIDEO_FPS),
        max_seconds=MAX_SECONDS,
        max_frames=MAX_FRAMES,
    )
    source_label = INPUT_VIDEO.name
else:
    raw_frames, image_files = load_image_frames(INPUT_DIR, FILE_GLOB)
    source_info = {"count": len(raw_frames)}
    source_label = f"{INPUT_DIR.name}/{FILE_GLOB}"

raw_frames = raw_frames[:MAX_FRAMES]
raw_frames = maybe_bounce(raw_frames)

print("Source        :", source_label)
print("Raw frame count:", len(raw_frames))
print("First frame size:", raw_frames[0].size)

In [ ]:
# -----------------------------
# Prepare GIF and MP4 frame sets
# -----------------------------
gif_frames = preprocess_frames(
    raw_frames,
    target_width=GIF_WIDTH,
    aspect_preset=ASPECT_PRESET,
    center_crop=CENTER_CROP,
    pad_color=PAD_COLOR,
    top_text=TOP_TEXT,
    bottom_text=BOTTOM_TEXT,
    ensure_even=False,
)

video_frames = preprocess_frames(
    raw_frames,
    target_width=VIDEO_WIDTH,
    aspect_preset=ASPECT_PRESET,
    center_crop=CENTER_CROP,
    pad_color=PAD_COLOR,
    top_text=TOP_TEXT,
    bottom_text=BOTTOM_TEXT,
    ensure_even=True,
)

print("GIF frame size :", gif_frames[0].size)
print("MP4 frame size :", video_frames[0].size)
print("GIF frames     :", len(gif_frames))
print("MP4 frames     :", len(video_frames))

In [ ]:
# -----------------------------
# Preview a few processed frames
# -----------------------------
preview_count = min(6, len(video_frames))
fig, axes = plt.subplots(1, preview_count, figsize=(3 * preview_count, 3))
if preview_count == 1:
    axes = [axes]

for ax, frame, idx in zip(axes, video_frames[:preview_count], range(preview_count)):
    ax.imshow(frame)
    ax.set_title(f"Frame {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------
# Export GIF / MP4
# -----------------------------
exported = {}

if wants_gif(MODE):
    gif_path = OUT_DIR / f"{BASENAME}_{ASPECT_PRESET}_{GIF_WIDTH}px.gif"
    write_gif(gif_frames, gif_path, fps=GIF_FPS, loop=GIF_LOOP, optimize=GIF_OPTIMIZE)
    exported["gif"] = gif_path
    print("Saved GIF:", gif_path)

if wants_mp4(MODE):
    mp4_path = OUT_DIR / f"{BASENAME}_{ASPECT_PRESET}_{VIDEO_WIDTH}px.mp4"
    write_mp4(video_frames, mp4_path, fps=VIDEO_FPS, codec=MP4_CODEC, quality=MP4_QUALITY)
    exported["mp4"] = mp4_path
    print("Saved MP4:", mp4_path)

if not exported:
    print("Nothing was exported. Check MODE.")

In [ ]:
# -----------------------------
# Inline preview
# -----------------------------
if HAS_IPYTHON_DISPLAY:
    if "gif" in exported:
        display(IPyImage(filename=str(exported["gif"])))
    if "mp4" in exported:
        display(IPyVideo(filename=str(exported["mp4"]), embed=True))
else:
    print("Notebook preview is unavailable in this environment.")

## Quick tweaks

### Make the file even smaller
- Lower `GIF_WIDTH` from `480` to `360`
- Lower `VIDEO_WIDTH` from `720` to `540`
- Reduce `MAX_SECONDS`
- Reduce `GIF_FPS` from `10` to `8`
- Reduce `VIDEO_FPS` from `24` to `20`

### Use it with a folder of screenshots or renders
- Put your frames inside `input_frames/`
- Set `MODE = "images_to_both"`
- Set `FILE_GLOB = "*.png"`

### Use it with a source video
- Put a clip at `input.mp4`
- Set `MODE = "video_to_both"`
- Adjust `MAX_SECONDS`

### Aspect presets
- `"landscape"` → YouTube-style 16:9
- `"square"` → simple channel promo or post
- `"portrait"` → shorts / reel style
- `"original"` → preserves source aspect ratio

## Done

Your exported files will be saved inside `OUT_DIR`.

Because this notebook uses a fixed execution order, it also avoids the kind of out-of-order helper issue that showed up in the pasted run.